# Loan Performance Intelligence Engine — End-to-End Walkthrough
### Intain Campus FinTech Challenge 2026 — AI Track

This notebook demonstrates the complete, reproducible pipeline across all 8 challenge tasks:
1. **Task 1**: Data Intelligence & Profiling
2. **Task 2**: Loan Performance Supervised Prediction & Time-Aware Validation
3. **Task 3**: Time-to-Event Survival & Transition Modeling
4. **Task 4**: Anomaly Detection & Exception Intelligence
5. **Task 5**: Macro Scenario & Stress Simulation
6. **Task 6**: Global & Local Explainability (SHAP)
7. **Task 7**: Grounded LLM Reviewer Copilot & Hallucination Audits
8. **Task 8**: Final Submission Scored Output

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import numpy as np
import pandas as pd
import json

print(f"Project root configured: {ROOT}")

## Phase 1: Load and Inspect Datasets

In [ ]:
train_path = ROOT / "data" / "raw" / "loan_monthly_performance_train.csv"
test_path = ROOT / "data" / "raw" / "loan_monthly_performance_test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f"Train panel shape: {train_df.shape} ({train_df['loan_id'].nunique():,} unique loans)")
print(f"Test panel shape:  {test_df.shape} ({test_df['loan_id'].nunique():,} unique loans)")
train_df.head(3)

## Task 1: Data Intelligence & Profiling

In [ ]:
from src.profiling.quality_scorer import evaluate_batch_quality
from src.profiling.missingness import analyze_missingness
from src.profiling.date_validator import validate_date_relationships

dq = evaluate_batch_quality(train_df)
print(f"Portfolio Data Quality Grade: {dq['batch_quality_grade']}")
print(f"Mean Data Quality Score: {dq['batch_mean_dq_score']:.2f} / 100.0")

missing_info = analyze_missingness(train_df)
print("\nMissingness mechanism classifications:")
for col, data in missing_info["mechanism_classification"].items():
    print(f" - {col}: {data['inferred_mechanism']}")

## Task 2: Feature Engineering & Time-Aware Validation Split

In [ ]:
from src.pipeline.splitter import time_aware_cohort_split, audit_split_leakage
from src.features.feature_engineer import engineer_panel_features, get_feature_columns

train_sub, val_sub, _ = time_aware_cohort_split(train_df, val_cutoff="2020-01-01", test_cutoff="2099-01-01")
audit = audit_split_leakage(train_sub, val_sub)
print(f"Leakage Audit Passed: {audit['is_leakage_free']} (Train-Val ID overlap: {audit['train_val_overlap_count']})")

feat_train = engineer_panel_features(train_sub)
feat_val = engineer_panel_features(val_sub)
print(f"Engineered {len(get_feature_columns())} non-leaking features.")

## Task 2 (cont.): Baseline vs Improved LightGBM Model Comparison

In [ ]:
from src.models.prediction.evaluate import run_full_evaluation

eval_results = run_full_evaluation(feat_val)
summary_df = pd.DataFrame(eval_results["summary_table"])
summary_df

## Task 3: Survival Modeling (Kaplan-Meier & Cox PH)

In [ ]:
from src.models.survival.kaplan_meier import prepare_survival_dataset, fit_kaplan_meier_curves
from src.models.survival.evaluate_survival import evaluate_survival_vs_baseline

surv_df = prepare_survival_dataset(train_df)
km_results = fit_kaplan_meier_curves(surv_df)
surv_eval = evaluate_survival_vs_baseline(surv_df)

print(f"Cox PH Concordance Index: {surv_eval['cox_ph_concordance_index']:.4f} (Lift: +{surv_eval['c_index_lift']:.4f})")
pd.DataFrame(surv_eval["horizon_comparisons"])

## Task 4: Anomaly Detection & Reviewer Cases

In [ ]:
from src.models.anomaly.explain_anomalies import generate_reviewer_anomaly_cases
from src.models.anomaly.isolation_forest import predict_anomaly_scores

feat_train["anomaly_score"] = predict_anomaly_scores(feat_train)
cases = generate_reviewer_anomaly_cases(feat_train, n_cases=5)
pd.DataFrame(cases)[["loan_id", "current_status", "current_balance", "anomaly_score", "exception_type", "plain_english_explanation"]]

## Task 5: Macro Scenario & Stress Simulation

In [ ]:
scen_json = ROOT / "src" / "models" / "saved_models" / "scenario_simulation_results.json"
if scen_json.exists():
    with open(scen_json) as f:
        scen_data = json.load(f)
    scen_rows = []
    for s in scen_data:
        p = s["portfolio_projected_rates"]
        scen_rows.append({
            "Scenario": s["scenario_name"],
            "3M Delinq (%)": p.get("mean_next_3m_delinquency_flag_rate"),
            "12M Default (%)": p.get("mean_next_12m_default_flag_rate"),
            "12M Prepayment (%)": p.get("mean_next_12m_prepayment_flag_rate"),
        })
    display(pd.DataFrame(scen_rows))
else:
    print("Run 'make scenarios' to generate scenario outputs.")

## Task 6: Explainability (TreeSHAP Local Waterfall)

In [ ]:
from src.explainability.local_explanation import explain_single_loan

sample_loan = feat_val.iloc[0]
local_exp = explain_single_loan(sample_loan, target="next_12m_default_flag")
print(f"Loan ID: {local_exp['loan_id']} | Predicted 12M Default Prob: {local_exp['predicted_probability']:.2%}")
print("\nTop Positive Risk Drivers (Increasing Risk):")
for d in local_exp["top_positive_risk_drivers"][:3]:
    print(f" - {d['feature']} = {d['feature_value']} (SHAP: +{d['shap_attribution']:.4f})")
print("\nTop Protective Drivers (Decreasing Risk):")
for d in local_exp["top_negative_protective_drivers"][:3]:
    print(f" - {d['feature']} = {d['feature_value']} (SHAP: {d['shap_attribution']:.4f})")

## Task 7: Grounded LLM Reviewer Copilot Demo

In [ ]:
from src.llm_copilot.copilot import GroundedReviewerCopilot

copilot = GroundedReviewerCopilot()
sample_payload = sample_loan.to_dict()
sample_preds = {
    "prob_next_3m_delinquency": 0.621,
    "prob_next_12m_default": 0.185,
    "prob_next_12m_prepayment": 0.045,
    "next_state": "30-59 DPD",
    "anomaly_score": 0.352,
    "top_drivers": ["days_past_due", "credit_score_ordinal"],
}

note = copilot.generate_reviewer_note(sample_payload, sample_preds)
print(note["reviewer_note"])

## Final Submission File Verification

In [ ]:
sub_path = ROOT / "submission" / "submission.csv"
if sub_path.exists():
    sub_df = pd.read_csv(sub_path)
    print(f"submission.csv verified: {len(sub_df):,} rows, {len(sub_df.columns)} columns")
    display(sub_df.head(5))
else:
    print("Run 'make submission' to generate final submission file.")